## Preperation of GEO GSE186328 data

In [ ]:
import time
from pathlib import Path

import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
from scipy import sparse

# =========================================================
# Paths
# =========================================================
base_dir = Path(r"path/to/your/directory")  # Change to your working directory
csv_path = base_dir / "GSE186328_raw_counts.csv"
output_h5ad = base_dir / "GSE186328_selected_raw.h5ad"

In [ ]:
# =========================================================
# GSE186328 Sample Information
# =========================================================
# 6 samples represented in GSE186328_raw_counts.csv by barcode prefixes:
# lv2, lv9, lv12, lv13, lv15, lv19

sample_prefixes = ["lv2", "lv9", "lv12", "lv13", "lv15", "lv19"]

print(f"[INFO] Looking for CSV at: {csv_path}")
print(f"[INFO] Expected sample prefixes ({len(sample_prefixes)}): {sample_prefixes}")

In [ ]:
# =========================================================
# Load raw counts CSV and construct AnnData object
# =========================================================
if not csv_path.exists():
    raise FileNotFoundError(f"CSV file not found at: {csv_path}")

print(f"[INFO] Reading CSV file: {csv_path} ...")
t0 = time.time()
df = pd.read_csv(csv_path, index_col=0)
t1 = time.time()
print(f"[INFO] CSV loaded in {t1 - t0:.2f} seconds.")
print(f"[INFO] Matrix shape (genes x cells): {df.shape}")

# Transpose matrix so shape becomes (n_cells x n_genes)
print("[INFO] Creating AnnData object with sparse CSR matrix...")
X = sparse.csr_matrix(df.T.values)

cell_ids = df.columns.tolist()
gene_names = df.index.astype(str).tolist()

obs = pd.DataFrame(index=cell_ids)
# Extract sample_id from cell barcode prefix (e.g., 'lv2_AAACCCAGTAACGATA' -> 'lv2')
obs["sample_id"] = [c.split("_")[0] for c in cell_ids]
obs["patient_id"] = obs["sample_id"]
obs["gsm_id"] = obs["sample_id"]

var = pd.DataFrame(index=gene_names)

adata = ad.AnnData(X=X, obs=obs, var=var)
adata.var_names_make_unique()
adata.obs_names_make_unique()

# Store raw counts in layers
if not sparse.issparse(adata.X):
    adata.X = sparse.csr_matrix(adata.X)

adata.layers["counts"] = adata.X.copy()

print("[INFO] AnnData creation complete.")

In [4]:
# =========================================================
# Inspect AnnData object
# =========================================================
print(adata)
print("\n[INFO] Cell count per sample:")
print(adata.obs["sample_id"].value_counts())

print("\n[INFO] Number of samples represented:")
print(adata.obs["sample_id"].nunique())

AnnData object with n_obs × n_vars = 47724 × 21650
    obs: 'sample_id', 'patient_id', 'gsm_id'
    layers: 'counts'

[INFO] Cell count per sample:
sample_id
lv12    10533
lv13    10272
lv2      8000
lv19     7627
lv15     5780
lv9      5512
Name: count, dtype: int64

[INFO] Number of samples represented:
6


In [ ]:
# =========================================================
# Save AnnData object to disk
# =========================================================
print(f"\n[INFO] Saving to: {output_h5ad}")
output_h5ad.parent.mkdir(parents=True, exist_ok=True)
adata_verify.write_h5ad(output_h5ad, compression="gzip")
print("[INFO] Done.")

In [39]:
import numpy as np
from scipy import sparse

def row_sums(M):
    return np.asarray(M.sum(axis=1)).ravel()

def expm1_row_sums(M):
    if sparse.issparse(M):
        M2 = M.copy()
        M2.data = np.expm1(M2.data)
        return np.asarray(M2.sum(axis=1)).ravel()
    else:
        return np.expm1(M).sum(axis=1)

def matrix_summary(M, name):
    data = M.data if sparse.issparse(M) else np.asarray(M).ravel()
    data = data[np.isfinite(data)]

    print(f"\n{name}")
    print("-" * len(name))
    print("shape:", M.shape)
    print("sparse:", sparse.issparse(M))
    print("min:", data.min())
    print("max:", data.max())
    print("mean nonzero/value:", data.mean())
    print("integer-like values:", np.mean(np.isclose(data, np.round(data))) > 0.99)

    rs = row_sums(M)
    print("row sum summary:")
    print(np.percentile(rs, [0, 1, 25, 50, 75, 99, 100]))

    ers = expm1_row_sums(M)
    print("expm1(row) sum summary:")
    print(np.percentile(ers, [0, 1, 25, 50, 75, 99, 100]))

    print("close to 10,000 after expm1?:", np.mean(np.isclose(ers, 10000, rtol=1e-3, atol=1)))



In [40]:
# print("============================")
# print(f"Looking at:")
# matrix_summary(adata.X, "adata.X")
# if "counts" in adata.layers:
#     matrix_summary(adata.layers["counts"], "adata.layers['counts']")

# if adata.raw is not None:
#     matrix_summary(adata.raw.X, "adata.raw.X")
print("============================")
print(f"Looking at:")
matrix_summary(adata_verify.X, "adata.X")
if "counts" in adata_verify.layers:
    matrix_summary(adata_verify.layers["counts"], "adata_verify.layers['counts']")

if adata_verify.raw is not None:
    matrix_summary(adata_verify.raw.X, "adata_verify.raw.X")

Looking at:

adata.X
-------
shape: (43031, 21181)
sparse: True
min: 1
max: 3202
mean nonzero/value: 2.7737698050393673
integer-like values: True
row sum summary:
[ 1002.   1078.   3249.   4492.   5861.5 11625.7 12405. ]


C:\Users\ankit\AppData\Local\Temp\ipykernel_19196\432754156.py:10: RuntimeWarning: overflow encountered in expm1
  M2.data = np.expm1(M2.data)
c:\Users\ankit\miniconda3\envs\scanpy\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a


expm1(row) sum summary:
[1.39956030e+005 1.15763451e+021 3.06984964e+069 3.65084638e+118
 1.18022415e+168             nan             nan]
close to 10,000 after expm1?: 0.0

adata_verify.layers['counts']
-----------------------------
shape: (43031, 21181)
sparse: True
min: 1
max: 3202
mean nonzero/value: 2.7737698050393673
integer-like values: True
row sum summary:
[ 1002.   1078.   3249.   4492.   5861.5 11625.7 12405. ]
expm1(row) sum summary:
[1.39956030e+005 1.15763451e+021 3.06984964e+069 3.65084638e+118
 1.18022415e+168             nan             nan]
close to 10,000 after expm1?: 0.0


In [ ]:
# =========================================================
# Read back saved h5ad file to verify
# =========================================================
print(f"\n[INFO] Reading back saved file: {output_h5ad}")
adata_verify = sc.read_h5ad(output_h5ad)
print(adata_verify)
print("\n[INFO] Sample cell counts from saved file:")
print(adata_verify.obs["sample_id"].value_counts())

In [ ]:
mapping = {
    "lv2": "Healthy",
    "lv9": "MASLD",
    "lv12": "MASLD",
    "lv13": "MASLD",
    "lv15": "Healthy",
    "lv19": "Healthy",
}

adata_verify.obs["broad_condition"] = adata_verify.obs["sample_id"].map(mapping).astype("category")
print(adata_verify.obs["broad_condition"].value_counts())


broad_condition
MASLD      26317
healthy    21407
Name: count, dtype: int64


In [8]:
mapping = {
    "lv2": "GSM5643817",
    "lv9": "GSM5643818",
    "lv12": "GSM5643819",
    "lv13": "GSM5643820",
    "lv15": "GSM5643821",
    "lv19": "GSM5643822",
}

In [9]:
adata_verify.obs["gsm_id"] = adata_verify.obs["sample_id"].map(mapping).astype("category")
print(adata_verify.obs["gsm_id"].value_counts())

gsm_id
GSM5643819    10533
GSM5643820    10272
GSM5643817     8000
GSM5643822     7627
GSM5643821     5780
GSM5643818     5512
Name: count, dtype: int64


In [25]:
# View the unique values in the 'gsm_id' column
print("\n[INFO] Unique values in 'gsm_id' column:")
print(adata_verify.obs["gsm_id"].unique())


[INFO] Unique values in 'gsm_id' column:
['GSM5643817', 'GSM5643818', 'GSM5643819', 'GSM5643820', 'GSM5643821', 'GSM5643822']
Categories (6, object): ['GSM5643817', 'GSM5643818', 'GSM5643819', 'GSM5643820', 'GSM5643821', 'GSM5643822']


In [23]:
# Add an "assay_type" column to the obs DataFrame
adata_verify.obs["assay_type"] = "scRNA-seq"

In [24]:
# Add a dataset column to the obs DataFrame
adata_verify.obs["dataset"] = "GSE186328"

In [22]:
adata_verify

AnnData object with n_obs × n_vars = 47724 × 21650
    obs: 'patient_id', 'gsm_id', 'broad_condition'
    layers: 'counts'

In [21]:
# Drop the 'sample_id' column if it exists
if "sample_id" in adata_verify.obs.columns:
    adata_verify.obs.drop(columns=["sample_id"], inplace=True)

## Filtering

In [27]:
from scipy import sparse
from scipy.stats import median_abs_deviation
import numpy as np

In [28]:
sc.pp.filter_cells(adata_verify, min_genes=100)
sc.pp.filter_genes(adata_verify, min_cells=3)

In [29]:

# Annotate genes for QC metrics.
adata_verify.var["mt"] = adata_verify.var_names.str.startswith("MT-")
adata_verify.var["ribo"] = adata_verify.var_names.str.startswith(("RPS", "RPL"))
adata_verify.var["hb"] = adata_verify.var_names.str.contains("^HB[^(P)]")

# Calculate QC metrics.
sc.pp.calculate_qc_metrics(
    adata_verify,
    qc_vars=["mt", "ribo", "hb"],
    inplace=True,
    percent_top=[20],
    log1p=True,
)



In [33]:
# Update n_counts, n_genes
adata_verify.obs["n_counts"] = adata_verify.X.sum(axis=1).A1 if sparse.issparse(adata_verify.X) else adata_verify.X.sum(axis=1)
print(f"adata did not have an n_counts column, so one was created by summing the adata.X matrix")

adata_verify.obs["n_genes"] = (adata_verify.X > 0).sum(axis=1).A1 if sparse.issparse(adata_verify.X) else (adata_verify.X > 0).sum(axis=1)
print(f"adata did not have an n_genes column, so one was created by summing the adata.X matrix")

adata did not have an n_counts column, so one was created by summing the adata.X matrix
adata did not have an n_genes column, so one was created by summing the adata.X matrix


In [34]:
# Pre filtering Mito %
print(adata_verify.shape)

adata_verify = adata_verify[adata_verify.obs["pct_counts_mt"] < 10, :]
adata_verify = adata_verify[adata_verify.obs["n_counts"] <= (np.median(adata_verify.obs["n_counts"]) + 5 * median_abs_deviation(adata_verify.obs["n_counts"])), :]

print("Post filtering mito %)")
print(adata_verify.shape)

(47724, 21181)
Post filtering mito %)
(43031, 21181)


In [ ]:
# =========================================================
# Save AnnData object to disk
# =========================================================
print(f"\n[INFO] Saving to: {output_h5ad}")
output_h5ad.parent.mkdir(parents=True, exist_ok=True)
adata_verify.write_h5ad(output_h5ad, compression="gzip")
print("[INFO] Done.")